# <center>Data and Artificial Intelligence</center>
## <center>Cyber Shujaa Program</center>
- Project: NLP with Transformers - Sentence Similarity with BERT

- Project Description: This project demonstrates the use of BERT for sentence similarity analysis

- Language: Python

- Author: Amidu Dabor | CS-DA01-25031

- Date: 28 July 2025

---

## Step 1: Import Required Libraries

In [1]:
from transformers import BertTokenizer, TFBertModel
import tensorflow as tf
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
from tabulate import tabulate

/opt/anaconda3/envs/venv_py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Step 2: Define Sentence Similarity Engine using BERT

In [2]:
# Define the BERT Similarity class
class BERTSimilarityEngine:
    def __init__(self, threshold=0.7):
        # Load tokenizer and model
        self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        self.model = TFBertModel.from_pretrained('bert-base-uncased')
        self.threshold = threshold

    def get_cls_embedding(self, sentence):
        # Tokenize and encode
        inputs = self.tokenizer(sentence, return_tensors='tf', padding=True, truncation=True)
        outputs = self.model(inputs)
        # Extract the [CLS] token (first position)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        return cls_embedding.numpy()

    def compute_similarity(self, sent1, sent2):
        # Get embeddings
        emb1 = self.get_cls_embedding(sent1)
        emb2 = self.get_cls_embedding(sent2)
        # Compute cosine similarity
        score = cosine_similarity(emb1, emb2)[0][0]
        # Predict label
        label = 1 if score > self.threshold else 0
        return score, label

    def evaluate(self, pairs, labels):
        predictions = []
        scores = []

        print("\n🔍 Sentence Similarity Evaluation:\n")
        for i, (s1, s2) in enumerate(pairs):
            score, pred = self.compute_similarity(s1, s2)
            scores.append(score)
            predictions.append(pred)
            print(f"Pair {i+1}")
            print(f"Sentence 1: {s1}")
            print(f"Sentence 2: {s2}")
            print(f"Cosine Similarity: {score:.4f} → Predicted: {'Similar' if pred else 'Not Similar'}\n")

        correct = sum([1 for p, l in zip(predictions, labels) if p == l])
        accuracy = correct / len(labels)
        print(f"Final Accuracy: {accuracy:.2%}")
        return predictions, accuracy, scores

## Step 3: Define Sentence Pairs and Labels

In [3]:
sentence_pairs = [
    ("How do I learn Python?", "What is the best way to study Python?"),
    ("What is AI?", "How to cook pasta?"),
    ("How do I bake a chocolate cake?", "Give me a chocolate cake recipe."),
    ("How can I improve my coding skills?", "Tips for becoming better at programming."),
    ("Where can I buy cheap laptops?", "Best sites to find affordable computers."),
    ("I love eating mangoes in summer.", "Mangoes taste best during hot seasons."),
    ("The bank will close at 5 PM today.", "He sat by the river bank and read."),
    ("She made a quick run to the store.", "She sprinted fast to buy groceries."),
    ("The crane lifted the heavy beam.", "I saw a crane flying over the lake."),
    ("He scored a goal in the final match.", "She submitted her final thesis."),
    ("The sun sets over the horizon.", "The moon rises from the sky."),
    ("He is planning a trip to the mountains.", "She is preparing a dinner meal."),
    ("I'm waiting for the bus to arrive.", "I'm waiting for the train to arrive."),
    ("She is making a cup of tea.", "He is making a cake."),
    ("She is a mother to two young children.", "He is a father to a young child.")
]

# Ground Truth Labels: 1 = Similar, 0 = Not Similar
# labels = [1, 0, 1, 1, 1, 1, 0, 1, 0, 0]
labels = [1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1]

## Step 4: Instantiate and Evaluate Model

In [4]:
# Instantiate and evaluate using model
engine = BERTSimilarityEngine()
predictions, accuracy, scores = engine.evaluate(sentence_pairs, labels)

# Prepare DataFrame
df = pd.DataFrame({
    "Pair ID": list(range(1, len(sentence_pairs) + 1)),
    "Sentence 1": [s[0] for s in sentence_pairs],
    "Sentence 2": [s[1] for s in sentence_pairs],
    "Cosine Similarity": scores,
    "Predicted": predictions,
    "Ground Truth": labels
})

df["Sentence Pair"] = df["Sentence 1"] + " ↔ " + df["Sentence 2"]

# Plotly figure using go.Figure - for better layout control
fig = go.Figure()

# Cosine Similarity Points (Blue Markers)
fig.add_trace(go.Scatter(
    x=df["Pair ID"],
    y=df["Cosine Similarity"],
    mode='markers+text',
    name='Cosine Similarity',
    marker=dict(color='blue', size=10),
    text=df["Pair ID"],
    textposition='top center',
    hovertemplate=
        "<b>Pair ID:</b> %{x}<br>" +
        "<b>Sentence Pair:</b><br>%{customdata[0]}<br>" +
        "<b>Cosine Similarity:</b> %{y:.4f}<br>" +
        "<b>Predicted:</b> %{customdata[1]}<br>" +
        "<b>Ground Truth:</b> %{customdata[2]}<extra></extra>",
    customdata=df[["Sentence Pair", "Predicted", "Ground Truth"]]
))

# Threshold Line (Red Dashed)
x_start = 0  # Start just before first point
x_end = df["Pair ID"].max() + 1  # End just after last point

fig.add_trace(go.Scatter(
    x=[x_start, x_end],
    y=[0.7, 0.7],
    mode='lines',
    line=dict(color='red', width=2, dash='dash'),
    name='Threshold = 0.7',
    hoverinfo='skip'
))

# # Add Threshold Line
# fig.add_hline(
#     y=0.7,
#     line_dash="dash",
#     line_color="red",
#     annotation_text="Threshold = 0.7",
#     annotation_position="top left",
#     name="Threshold"
# )

# Layout & Style
fig.update_layout(
    title="Sentence Pair Similarity Scores with Predictions",
    xaxis_title="Sentence Pair ID",
    yaxis_title="Cosine Similarity",
    height=700,
    width=1100,
    yaxis=dict(range=[0, 1.05]),
    xaxis=dict(tickmode='linear'),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="center",
        x=0.5
    ),
    hoverlabel=dict(bgcolor="white", font_size=13)
)

fig.show()

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.seq_relationship.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were 


🔍 Sentence Similarity Evaluation:

Pair 1
Sentence 1: How do I learn Python?
Sentence 2: What is the best way to study Python?
Cosine Similarity: 0.9743 → Predicted: Similar

Pair 2
Sentence 1: What is AI?
Sentence 2: How to cook pasta?
Cosine Similarity: 0.9033 → Predicted: Similar

Pair 3
Sentence 1: How do I bake a chocolate cake?
Sentence 2: Give me a chocolate cake recipe.
Cosine Similarity: 0.8938 → Predicted: Similar

Pair 4
Sentence 1: How can I improve my coding skills?
Sentence 2: Tips for becoming better at programming.
Cosine Similarity: 0.8633 → Predicted: Similar

Pair 5
Sentence 1: Where can I buy cheap laptops?
Sentence 2: Best sites to find affordable computers.
Cosine Similarity: 0.8750 → Predicted: Similar

Pair 6
Sentence 1: I love eating mangoes in summer.
Sentence 2: Mangoes taste best during hot seasons.
Cosine Similarity: 0.8350 → Predicted: Similar

Pair 7
Sentence 1: The bank will close at 5 PM today.
Sentence 2: He sat by the river bank and read.
Cosine Simi

In [6]:
engine.get_cls_embedding(sentence_pairs[0][0])

array([[-3.91852036e-02,  2.00920671e-01,  1.42418295e-02,
        -4.57232669e-02, -2.91943014e-01,  1.55994594e-02,
         7.30915546e-01,  4.47535932e-01, -2.41496414e-01,
         1.58486843e-01, -7.40697011e-02, -1.75546169e-01,
        -1.09237850e-01, -1.04426518e-02,  3.90605181e-01,
        -8.40551257e-02, -1.34050623e-01,  3.91230404e-01,
         2.96944469e-01,  2.25489885e-01,  8.91168416e-02,
         2.85556726e-03, -1.59137815e-01, -2.80068778e-02,
         1.36740595e-01, -9.25886706e-02,  1.31215885e-01,
         9.62996036e-02,  2.41178200e-02, -1.74419105e-01,
         1.36259198e-01,  7.11333901e-02, -1.05665490e-01,
        -2.79373854e-01,  1.78407237e-01, -1.54735342e-01,
        -9.88034979e-02, -1.42643452e-01,  8.13576728e-02,
         7.21337497e-02, -3.76233995e-01,  1.09871104e-01,
        -2.92348564e-01,  1.76500693e-01,  1.66223750e-01,
        -4.67850655e-01, -2.22842360e+00, -4.50845718e-01,
        -4.83272284e-01, -1.96760133e-01, -3.71105336e-0

In [7]:
engine.get_cls_embedding("She made a quick run to the store.")

array([[ 1.94903597e-01, -8.26063529e-02, -1.97548747e-01,
         1.31812170e-02,  1.44846231e-01, -3.12832922e-01,
         2.35954672e-03,  6.52519763e-01,  2.61001348e-01,
        -2.40377009e-01,  1.18824050e-01, -1.41004890e-01,
         5.80372922e-02, -1.72182962e-01,  2.11199209e-01,
        -7.39363432e-02, -2.93122649e-01, -8.09094831e-02,
        -7.20383599e-03, -2.65150145e-02, -9.18813348e-02,
        -3.73964906e-01,  1.79698765e-01,  5.36940038e-01,
         2.98572063e-01, -3.64454314e-02, -1.88356414e-01,
         2.11771838e-02,  3.29418331e-01,  1.14464566e-01,
        -1.97535262e-01,  2.68949509e-01, -4.14146692e-01,
        -2.58391112e-01,  1.45895422e-01, -1.34626463e-01,
         3.75526279e-01, -8.25229660e-02,  3.13938186e-02,
         2.91352630e-01, -8.17624807e-01, -3.48675579e-01,
        -5.37271239e-02,  2.24469811e-01,  4.03735995e-01,
        -1.83318853e-01, -3.18210530e+00, -1.77990198e-01,
        -9.38765332e-02, -3.92639637e-01,  2.63747185e-0

---

## Step 5: Conceptual Questions

### 1. How does BERT differ from Bag-of-Words or TF-IDF?
- BoW and TF-IDF produce static embeddings, ignoring word order and context.
- BERT generates **contextual embeddings** that consider sentence structure and meaning, enabling it to distinguish polysemy.

### 2. What is the Role of the Encoder in BERT?
- BERT's encoder is a stack of Transformer layers that apply self-attention to capture relationships between words.
- In this assignment, we use the encoder's [CLS] token output for sentence embeddings.

### 3. What are Contextual Embeddings?
- Contextual embeddings vary depending on the sentence.
- "Bank" in "river bank" ≠ "Bank" in "finance".
- BERT captures this by processing the entire sequence together.

### 4. Why Use the [CLS] Token?
- It is a special classification token placed at the start of the input.
- BERT is trained so that its [CLS] output summarizes the entire input sentence.

### 5. What is Cosine Similarity?
- Cosine similarity measures the angle between two vectors.
- A score of 1 means identical direction (similar), 0 means orthogonal (unrelated).
- Useful for comparing sentence embeddings semantically.

---
